# Testing the `Intent_classification` n8n Workflow

Test cases are loaded from `unit_tests/classification/*/test.json`.
Each file contains a pre-built `input` dict and an `expectedOutput` with
`route_to` and a reference `narrative`.

### Routing values
`advisor` | `education` | `summary` | `unknown`

### Request payload shape
| Field | Meaning |
|---|---|
| `userMessage` | Latest message from the customer |
| `LastAImessage` | The previous agent's reply (if any) |
| `PrevAIagent` | Name of the agent that produced `LastAImessage` |
| `narrative` | Running narrative carried over from prior turns |

### URL note
- Test URL (n8n editor open): `{base}/webhook-test/{path}`
- Production URL (workflow Active): `{base}/webhook/{path}`

In [1]:
import json
from pathlib import Path
import glob

import pandas as pd
import requests

pd.set_option('display.max_rows', 2000)
pd.set_option('display.max_columns', 1500)

## 1. Configuration

In [2]:
N8N_BASE_URL = "https://alphamakeathon-automation.arisetech.dev"  # <-- change me
WEBHOOK_PATH = "dbce5b9e-1397-459a-871a-5b27433f1640"
USE_TEST_URL = False  # True -> use the "Listen for test event" URL instead


def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


get_webhook_url()

'https://alphamakeathon-automation.arisetech.dev/webhook/dbce5b9e-1397-459a-871a-5b27433f1640'

## 2. Load test cases from JSON

In [3]:
def _find_project_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "unit_tests").is_dir():
            return p
        p = p.parent
    raise RuntimeError("Could not find project root (directory containing 'unit_tests/')")


def load_unit_tests(agent_type: str) -> list[dict]:
    root = _find_project_root()
    test_dir = root / "unit_tests" / agent_type
    tests = []
    for tc_dir in sorted(test_dir.iterdir()):
        test_file = tc_dir / "test.json"
        if test_file.is_file():
            with open(test_file, encoding="utf-8") as f:
                tests.append(json.load(f))
    print(f"Loaded {len(tests)} test cases from {test_dir}")
    return tests


TEST_CASES = load_unit_tests("classification")
for tc in TEST_CASES:
    print(f"  {tc['testId']}: {tc['testDescription']}")

Loaded 100 test cases from /Users/IF668129/Downloads/DebtMind-test-case-generator-main/final_test_case_gen/unit_tests/classification
  TC-0001: [strict] MULTI_TURN — expected route_to=advisor
  TC-0002: [strict] STAFF_REQUEST — expected route_to=advisor
  TC-0003: [creative] NEW_CONVERSATION — expected route_to=unknown
  TC-0004: [adversarial] NEW_CONVERSATION — expected route_to=advisor
  TC-0005: [strict] OFF_TOPIC — expected route_to=unknown
  TC-0006: [strict] OFF_TOPIC — expected route_to=unknown
  TC-0007: [adversarial] NEGOTIATE_PAYMENT — expected route_to=advisor
  TC-0008: [strict] MULTI_TURN — expected route_to=education
  TC-0009: [strict] OFF_TOPIC — expected route_to=unknown
  TC-0010: [strict] MULTI_TURN — expected route_to=education
  TC-0011: [creative] EDUCATION_QUESTION — expected route_to=advisor
  TC-0012: [creative] NEGOTIATE_PAYMENT — expected route_to=advisor
  TC-0013: [adversarial] NEGOTIATE_TERM — expected route_to=advisor
  TC-0014: [adversarial] NEGOTIATE_TE

## 3. Webhook caller

In [4]:
def call_intent_classifier(payload: dict, timeout: int = 30) -> dict:
    """POST the pre-built input dict to the Intent Classification webhook."""
    url = get_webhook_url()
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

## 4. Run a single example

Inspect one test case and confirm connectivity before running the full suite.

In [5]:
sample_tc = TEST_CASES[0]
print(f"Test: {sample_tc['testId']} — {sample_tc['testDescription']}")
print("\nInput payload:")
print(json.dumps(sample_tc["input"], ensure_ascii=False, indent=2))
print("\nExpected output:")
print(json.dumps(sample_tc["expectedOutput"], ensure_ascii=False, indent=2))

# Uncomment once N8N_BASE_URL points to a real, reachable instance:
# result = call_intent_classifier(sample_tc["input"])
# print("\nActual output:")
# print(json.dumps(result, ensure_ascii=False, indent=2))

Test: TC-0001 — [strict] MULTI_TURN — expected route_to=advisor

Input payload:
{
  "userMessage": "เปลี่ยนใจละ อยากผ่อนมากกว่านี้",
  "LastAImessage": "เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข้อมูลดังนี้\n\nแผนที่ 1: TDR ด้วยอัตรากำลังผ่อนชำระของลูกค้า\n  - รูปแบบ: ด้วยอัตรากำลังผ่อนชำระของลูกค้า\n  - บัญชี: xxxxxx501645, xxxxxx501646\n  - ระยะเวลา: 41 งวด\n  - ค่างวด: 396.47 บาท\n  - ดอกเบี้ยรวม: 6553.31 บาท\n\nแผนที่ 2: TDR ด้วยอัตรากำลังผ่อนชำระของลูกค้าและกระแสเงินสด\n  - รูปแบบ: ด้วยอัตรากำลังผ่อนชำระของลูกค้าและกระแสเงินสด\n  - บัญชี: xxxxxx501645, xxxxxx501646\n  - ระยะเวลา: 16 งวด\n  - ค่างวด: 396.47 บาท\n  - ดอกเบี้ยรวม: 3653.91 บาท",
  "PrevAIagent": "Advisor",
  "narrative": "The advisor presented two TDR plans with an 800 baht monthly installment. The customer has reviewed the options and now wishes to modify the plan to pay a higher monthly amount. The conversation is being routed back to the advisor to generate a new plan."
}

Expected output:
{
  "route_to": "advi

## 5. Run all test cases and compare `route_to`

In [6]:
rows = []
for tc in TEST_CASES:
    expected = tc["expectedOutput"]
    expected_route = expected.get("route_to")
    try:
        result = call_intent_classifier(tc["input"])
        actual_route = result.get("route_to")
        actual_narrative = result.get("narrative", "")
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "expected_route": expected_route,
            "actual_route": actual_route,
            "route_match": actual_route == expected_route,
            "actual_narrative": actual_narrative,
            "error": None,
        })
    except requests.exceptions.RequestException as exc:
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "expected_route": expected_route,
            "actual_route": None,
            "route_match": False,
            "actual_narrative": None,
            "error": str(exc),
        })

results_df = pd.DataFrame(rows)
summary_cols = ["testId", "scenario", "messageMode", "expected_route", "actual_route", "route_match", "error"]
print(f"Passed: {results_df['route_match'].sum()}/{len(results_df)}")
results_df[summary_cols]

Passed: 88/100


,testId,scenario,messageMode,expected_route,actual_route,route_match,error
0,TC-0001,MULTI_TURN,strict,advisor,advisor,True,None
1,TC-0002,STAFF_REQUEST,strict,advisor,advisor,True,None
2,TC-0003,NEW_CONVERSATION,creative,unknown,unknown,True,None
3,TC-0004,NEW_CONVERSATION,adversarial,advisor,advisor,True,None
4,TC-0005,OFF_TOPIC,strict,unknown,unknown,True,None
5,TC-0006,OFF_TOPIC,strict,unknown,unknown,True,None
6,TC-0007,NEGOTIATE_PAYMENT,adversarial,advisor,advisor,True,None
7,TC-0008,MULTI_TURN,strict,education,education,True,None
8,TC-0009,OFF_TOPIC,strict,unknown,unknown,True,None
9,TC-0010,MULTI_TURN,strict,education,education,True,None


In [7]:
results_df.head()

,testId,scenario,messageMode,expected_route,actual_route,route_match,actual_narrative,error
0,TC-0001,MULTI_TURN,strict,advisor,advisor,True,The advisor presented two TDR (debt restructur...,None
1,TC-0002,STAFF_REQUEST,strict,advisor,advisor,True,The customer is requesting a consultation with...,None
2,TC-0003,NEW_CONVERSATION,creative,unknown,unknown,True,The user is questioning the service's capabili...,None
3,TC-0004,NEW_CONVERSATION,adversarial,advisor,advisor,True,The user wants to restructure their debt and i...,None
4,TC-0005,OFF_TOPIC,strict,unknown,unknown,True,The user began the conversation by asking for ...,None


In [8]:
# df_testid = pd.read_excel("test_cases_classification.xlsx")
# df_testid.head()

In [9]:
# df_testid.shape

In [10]:
# results_df2 = results_df[results_df["testId"].isin(df_testid["testId"].tolist())]
results_df2 = results_df.copy()
results_df2.shape

(100, 8)

In [11]:
results_df2.head()

,testId,scenario,messageMode,expected_route,actual_route,route_match,actual_narrative,error
0,TC-0001,MULTI_TURN,strict,advisor,advisor,True,The advisor presented two TDR (debt restructur...,None
1,TC-0002,STAFF_REQUEST,strict,advisor,advisor,True,The customer is requesting a consultation with...,None
2,TC-0003,NEW_CONVERSATION,creative,unknown,unknown,True,The user is questioning the service's capabili...,None
3,TC-0004,NEW_CONVERSATION,adversarial,advisor,advisor,True,The user wants to restructure their debt and i...,None
4,TC-0005,OFF_TOPIC,strict,unknown,unknown,True,The user began the conversation by asking for ...,None


In [12]:
results_df2['route_match'].value_counts()

route_match
True     88
False    12
Name: count, dtype: int64

In [13]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

ROUTES = ["advisor", "education", "summary", "unknown"]

def evaluate_routes(results_df2: pd.DataFrame) -> pd.DataFrame:

    rows = []

    for route in ROUTES:
        y_true = results_df2["expected_route"].eq(route)
        y_pred = results_df2["actual_route"].eq(route)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[False, True]
        ).ravel()

        rows.append({
            "route": route,
            "support": len(results_df2),
            "positives": int(y_true.sum()),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "accuracy": round(accuracy_score(y_true, y_pred), 4),
            "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
            "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
            "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
            "specificity": round(
                tn / (tn + fp), 4
            ) if (tn + fp) > 0 else float("nan"),
        })

    metrics_df = pd.DataFrame(rows)

    exact_match = (results_df2["expected_route"] == results_df2["actual_route"])

    print(
        f"Overall Accuracy: "
        f"{exact_match.mean():.4f} "
        f"({exact_match.sum()}/{len(results_df2)})"
    )

    return metrics_df

In [14]:
metrics_df = evaluate_routes(results_df2)
display(metrics_df)

Overall Accuracy: 0.8800 (88/100)


,route,support,positives,TP,FP,FN,TN,accuracy,precision,recall,f1,specificity
0,advisor,100,65,56,3,9,32,0.88,0.9492,0.8615,0.9032,0.9143
1,education,100,14,14,3,0,83,0.97,0.8235,1.0000,0.9032,0.9651
2,summary,100,11,8,6,3,83,0.91,0.5714,0.7273,0.6400,0.9326
3,unknown,100,10,10,0,0,90,1.00,1.0000,1.0000,1.0000,1.0000


In [15]:
from sklearn.metrics import confusion_matrix

cm = pd.DataFrame(
    confusion_matrix(
        results_df2["expected_route"],
        results_df2["actual_route"],
        labels=ROUTES
    ),
    index=[f"true_{r}" for r in ROUTES],
    columns=[f"pred_{r}" for r in ROUTES],
)

display(cm)

,pred_advisor,pred_education,pred_summary,pred_unknown
true_advisor,56,3,6,0
true_education,0,14,0,0
true_summary,3,0,8,0
true_unknown,0,0,0,10


In [16]:
results_df2[results_df2['route_match'] == False]

,testId,scenario,messageMode,expected_route,actual_route,route_match,actual_narrative,error
30,TC-0031,STAFF_REQUEST,strict,advisor,summary,False,The advisor presented two TDR plans. The custo...,None
33,TC-0034,STAFF_REQUEST,creative,advisor,summary,False,The user is asking for clarification on the pr...,None
56,TC-0057,STAFF_REQUEST,adversarial,advisor,summary,False,The advisor has presented several TDR plans. T...,None
63,TC-0064,STAFF_REQUEST,strict,advisor,summary,False,The user has reviewed the proposed debt restru...,None
68,TC-0069,STAFF_REQUEST,creative,advisor,summary,False,The customer is confused by the bot's proposed...,None
74,TC-0075,NEGOTIATE_TERM,strict,advisor,education,False,The user was presented with a specific restruc...,None
75,TC-0076,AFTER_OFFER_TEXT,strict,advisor,education,False,The advisor presented two debt restructuring p...,None
78,TC-0079,MULTI_TURN,strict,advisor,education,False,The system presented the customer's personal l...,None
80,TC-0081,AFTER_OFFER_TEXT,creative,advisor,summary,False,The customer has reviewed the debt restructuri...,None
83,TC-0084,AFTER_OFFER_TEXT,creative,summary,advisor,False,The advisor offered a debt restructuring plan....,None


In [17]:
df_testid = pd.read_excel("test_case_classification.xlsx")
# df_testid.head()

In [18]:
failed_test_ids = results_df2.loc[~results_df2["route_match"], "testId"]
df_failed = df_testid[df_testid["testId"].isin(failed_test_ids)]

# add actual_route
df_failed = df_failed.merge(results_df2.loc[~results_df2["route_match"], ["testId", "actual_route"]], on="testId", how="left")

df_failed

,testId,scenario,messageMode,input_userMessage,input_lastAIMessage,input_prevAIAgent,input_narrative,expected_route_to,expected_narrative,actual_route
0,TC-0031,STAFF_REQUEST,strict,ดิฉันขอคุยกับเจ้าหน้าที่เพื่อหารือเรื่องแผนการ...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,The advisor presented two TDR plans with an 80...,advisor,The advisor previously presented two TDR restr...,summary
1,TC-0034,STAFF_REQUEST,creative,ข้อเสนอคืออะไรคะ 966 บาท? คืออยากรวมหนี้ทุกอย่...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,The user wants to restructure their personal l...,advisor,The customer is seeking clarification about th...,summary
2,TC-0057,STAFF_REQUEST,adversarial,ขอคุยกับเจ้าหน้าที่หน่อยค่ะ พอดีเป็นพนักงานสาข...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,The advisor presented two TDR plan options. Th...,advisor,The advisor previously presented multiple debt...,summary
3,TC-0064,STAFF_REQUEST,strict,ฉันได้ดูแผนที่เสนอมาแล้วค่ะ แต่อยากปรับแผนผ่อน...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,The advisor presented two TDR plans with an 80...,advisor,The customer has reviewed the restructuring pl...,summary
4,TC-0069,STAFF_REQUEST,creative,บอทงงอะไรอยู่คะ? เมื่อกี้พนักงานบอก 800 แต่บอท...,เสนอแผนการปรับโครงสร้างสินเชื่อ ด้วยแผนงานและข...,ADVISOR,The advisor presented two TDR plans with an 80...,advisor,The customer is confused by the discrepancy be...,summary
5,TC-0075,NEGOTIATE_TERM,strict,มีโปรแกรมอื่นนอกจาก PL_MOU ไหมครับ,เสนอแนวทางการปรับปรุงสินเชื่อ ด้วยแนวทางและข้อ...,advisor,Customer is exploring other restructuring prog...,advisor,different program,education
6,TC-0076,AFTER_OFFER_TEXT,strict,สองแผนนี้ต่างกันยังไงครับ,เสนอแนวทางการปรับปรุงสินเชื่อ ด้วยแนวทางและข้อ...,advisor,Customer is asking for differences between the...,advisor,compare plans,education
7,TC-0079,MULTI_TURN,strict,บัญชี 10000004 มีรายละเอียดเพิ่มเติมไหมครับ,ยินดีให้บริการคำปรึกษาแก้ปัญหาหนี้ค่ะ เพื่อให้...,advisor,The customer expressed their interests to find...,advisor,account detail,education
8,TC-0081,AFTER_OFFER_TEXT,creative,โอเคแต่ขอถามก่อนนะครับ ดอกเบี้ยยังปรับได้ไหม,เสนอแนวทางการปรับปรุงสินเชื่อ ด้วยแนวทางและข้อ...,advisor,Customer reviewed the proposed restructuring p...,advisor,Customer expressed tentative agreement but ask...,summary
9,TC-0084,AFTER_OFFER_TEXT,creative,โอเคเลย เอาแผนนี้แหละ,เสนอแนวทางการปรับปรุงสินเชื่อ ด้วยแนวทางและข้อ...,advisor,Customer requested debt restructuring plan.,summary,The customer has explicitly agreed to accept t...,advisor


## Notes

- This workflow is **Active**, so the production URL is
  `{base_url}/webhook/dbce5b9e-1397-459a-871a-5b27433f1640`.
- `route_to` is compared exactly; `narrative` is shown for inspection but not
  used for pass/fail since it is LLM-generated free text.
- Add more test cases by running the unit-test generator; they are picked up
  automatically the next time this notebook runs.
- If the n8n webhook requires authentication, add `auth=` or `headers=` to
  `requests.post` inside `call_intent_classifier`.